# Поиск по документам: TF-IDF и SVD

На прошлом занятии сообщение получало название отдела. Теперь пользователю нужен конкретный документ: например, инструкция по продлению книги. Построим два варианта поиска по одной базе и посмотрим, когда изменение представления меняет порядок результатов.

В базе двенадцать документов. Вопросы хранятся отдельно; часть из них относится к сведениям, которых в документах нет.

In [ ]:
from pathlib import Path
import os
import json
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Находим корень раздачи независимо от места запуска Jupyter.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data" / "requests.csv").exists())
DATA = ROOT / "data"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(42)
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.figsize": (8, 4)})


## Что будем искать

Откройте список документов и выберите тот, в котором ожидаете найти условия продления книги. Позже сравним это предположение с выдачей. Поле `relevant_doc` в таблице вопросов содержит эталонный документ; пустая строка означает, что ответа в базе нет. Эти метки используются при оценке, а не при построении индекса.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

docs = pd.read_csv(DATA / "knowledge.csv", keep_default_na=False)
queries = pd.read_csv(DATA / "retrieval_queries.csv", keep_default_na=False)
content = docs["title"] + ". " + docs["text"]
doc_ids = docs["doc_id"].to_numpy()
assert docs["doc_id"].is_unique
assert not {"relevant_doc", "reference_answer"} & set(docs.columns)
print("Документов:", len(docs), "Вопросов:", len(queries))
display(docs[["doc_id", "title"]])

### Один расчёт перед функцией

Пусть запрос представлен вектором (1;0), а документ — (1;1). Их скалярное произведение равно 1, а произведение длин равно √2. Следующая ячейка воспроизводит расчёт. Увеличьте второй вектор в десять раз и сначала предскажите результат.

In [ ]:
demo_query = np.array([1., 0.])
demo_document = np.array([1., 1.])
print(np.dot(demo_query, demo_document) / (np.linalg.norm(demo_query) * np.linalg.norm(demo_document)))

## 1. Сходство направлений

Напишите `cosine_safe(a, b)` для одномерных векторов одинаковой размерности. Используйте скалярное произведение и евклидовы нормы. Если размерности не совпадают или одна из норм равна нулю, вызывайте `ValueError`.

Сравните (1;0) с (1;1), (0;1), (10;10) и (−1;0). Объясните, почему положительный масштаб не влияет на косинус, а отрицательный меняет его знак. Можно ли прочитать число 0,7071 как вероятность правильного ответа?

In [ ]:
def cosine_safe(a, b):
    # Ваш код
    raise NotImplementedError


In [ ]:
assert np.isclose(cosine_safe([1, 0], [1, 1]), 1 / np.sqrt(2))
assert np.isclose(cosine_safe([1, 0], [10, 10]), cosine_safe([1, 0], [1, 1]))
try:
    cosine_safe([0, 0], [1, 0])
except ValueError:
    pass
else:
    raise AssertionError("Нужна явная обработка нулевого вектора")

## 2. Два представления одной базы

Обучите `vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm="l2")` на `content` и сохраните матрицу в `X_docs`. Затем обучите `svd = TruncatedSVD(n_components=6, random_state=42)` на этой матрице. Нормируйте полученные строки через `normalize` и сохраните их в `Z_docs`.

Сравните формы матриц и количество ненулевых элементов. Объясните, что теперь означает строка и что изменилось в её координатах. Почему для индекса можно использовать все известные документы, но нельзя добавлять в обучение проверочные вопросы с ответами?

In [ ]:
# Ваш код
vectorizer = None
svd = None
X_docs = None
Z_docs = None
raise NotImplementedError

In [ ]:
assert X_docs.shape[0] == len(docs)
assert Z_docs.shape == (len(docs), 6)
assert np.allclose(np.linalg.norm(Z_docs, axis=1), 1)
assert svd.components_.shape[1] == len(vectorizer.vocabulary_)

## 3. От запроса к выдаче

Закончите `search(query, method="tfidf", k=3)`. Запрос должен проходить через уже обученные `vectorizer` и, для метода `svd`, `svd` с последующей нормировкой. Сравните его с соответствующей матрицей документов.

Верните DataFrame с колонками `doc_id`, `title`, `score`: сначала большая неокруглённая оценка, при равенстве — меньший `doc_id`. Для нулевого вектора запроса верните пустую таблицу с теми же колонками. Проверьте оба метода на вопросе «Как продлить книгу?», пустой строке и строке «щщщщщщщ».

Почему новое `fit` на одном запросе испортило бы сравнение, даже если удалось получить массив нужного размера?

In [ ]:
def search(query, method="tfidf", k=3):
    # Ваш код
    raise NotImplementedError


In [ ]:
for method in ["tfidf", "svd"]:
    r = search("Как продлить книгу?", method, 3)
    assert list(r.columns) == ["doc_id", "title", "score"]
    assert len(r) <= 3 and r["doc_id"].is_unique
    assert r["score"].is_monotonic_decreasing
    assert search("", method).empty
    assert search("щщщщщщщ", method).empty

## 4. Считаем попадания и ранги

Реализуйте `evaluate(part, method)`. В среднее включайте только вопросы с непустым `relevant_doc`. Возвращайте словарь с ключами `recall_at_1`, `recall_at_3`, `mrr_at_3`, `n_answerable`. Если эталон отсутствует в первых трёх результатах, его вклад в MRR@3 равен нулю.

Соберите `results_table` с колонками `split`, `method` и четырьмя показателями. Сравните заранее выбранные TF-IDF и SVD(6) на validation, затем на test. После просмотра теста оставьте настройки неизменными.

Проверьте формулы на четырёх рангах из лекции: 1, 3, отсутствие в top-3 и 2. Чем отличаются Recall@3 и MRR@3 на этом примере?

In [ ]:
def evaluate(part, method):
    # Ваш код
    raise NotImplementedError

# Ваш код
raise NotImplementedError

In [ ]:
assert (results_table["recall_at_3"] >= results_table["recall_at_1"] - 1e-12).all()
assert results_table["mrr_at_3"].between(0, 1).all()
assert set(results_table["n_answerable"]) == {8, 16}
# Четыре ранга из лекции: 1, 3, отсутствие и 2.
assert np.isclose(np.mean([1, 1/3, 0, 1/2]), 11/24)


## Читаем различия выдач

Следующая ячейка собирает вопросы, для которых порядок документов отличается, и показывает оценки. Выберите два таких вопроса для разбора. Если документ попал в выдачу с нулевой оценкой, проверьте роль сортировки при равенстве.

После таблицы появятся вопросы без ответа в базе. Поиск всё равно может вернуть близкий документ. Прочитайте его и найдите, какого именно факта не хватает. Особенно полезен вопрос о сумме стипендии.

In [ ]:
differences = []
for row in queries[queries["relevant_doc"].ne("")].itertuples():
    first = search(row.query, "tfidf", 3)
    second = search(row.query, "svd", 3)
    if first["doc_id"].tolist() != second["doc_id"].tolist():
        differences.append({"query_id": row.query_id, "query": row.query,
                            "relevant_doc": row.relevant_doc,
                            "tfidf": list(zip(first["doc_id"], first["score"].round(6))),
                            "svd": list(zip(second["doc_id"], second["score"].round(6)))})
differences_table = pd.DataFrame(differences)
display(differences_table)

print("Вопросы без ответа в базе:")
for row in queries[queries["relevant_doc"].eq("")].itertuples():
    print(row.query)
    for method in ["tfidf", "svd"]:
        r = search(row.query, method, 1)
        print(method, r.to_dict("records"))

## Разбор выдачи

Выберите два вопроса, для которых методы дали разный порядок документов, и один вопрос без ответа в базе. Для каждого прочитайте найденные документы: где именно находится нужное сведение и какие совпадения могли повлиять на ранжирование? Покажите, как изменение ранга отражается на метриках. Если на выбранных вопросах выдачи совпали, возьмите другую строку из таблицы различий.

Сдайте выполненный ноутбук с этими разборами. Перед сдачей перезапустите ядро и выполните все ячейки по порядку.